In [1]:
# ===== Import libraries =====
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

# ===== Load dataset =====
url = "https://drive.google.com/file/d/1JPFEjSOlQ-gWExGiUv2kB1-yHWGfZdOQ/view?usp=sharing"
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]

housing = pd.read_csv(path)
housing.head()

# ===== Define features and target =====
X = housing.drop(columns=["SalePrice", "Id"], errors="ignore")
y = np.log1p(housing["SalePrice"])

# ===== Split data =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== Separate categorical and numerical =====
X_cat = X_train.select_dtypes(include=["object"])
X_num = X_train.select_dtypes(exclude=["object"])

# ===== Preprocessing =====
preprocessor = make_column_transformer(
    (
        make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        ),
        X_cat.columns
    ),
    (
        SimpleImputer(strategy="median"),
        X_num.columns
    ),
    remainder="drop"
)

# ===== Pipeline =====
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("feature_selection", SelectKBest(score_func=f_regression)),
    ("model", SGDRegressor(random_state=42, max_iter=2000, tol=1e-3))
])

# ===== Parameter grid =====
param_grid = {
    "feature_selection__k": [5, 10, 15, 20, 25, 30, "all"]
}

# ===== GridSearch with RMSE =====
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

# ===== Train =====
grid.fit(X_train, y_train)

# ===== Best model =====
best_model = grid.best_estimator_

print("Best k:", grid.best_params_["feature_selection__k"])
print("Best CV RMSE (log scale):", -grid.best_score_)

# ===== Predict =====
y_pred_log = best_model.predict(X_test)

# ===== Evaluation (log scale) =====
print("\n--- LOG SCALE ---")
print("Test RMSE:", mean_squared_error(y_test, y_pred_log) ** 0.5)
print("Test R2:", r2_score(y_test, y_pred_log))

# ===== Back to real scale =====
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)

print("\n--- ORIGINAL SCALE ---")
print("Test RMSE:", mean_squared_error(y_test_real, y_pred_real) ** 0.5)

Best k: 25
Best CV RMSE (log scale): 0.16239174386169467

--- LOG SCALE ---
Test RMSE: 0.15939031708974113
Test R2: 0.8638597149393081

--- ORIGINAL SCALE ---
Test RMSE: 29322.915689712536


#COMPETITION-Kaggle

In [2]:
# ===== Load teacher's external test data =====
url = "https://drive.google.com/file/d/1q14sdW_8Gk9x0h5fAejPpq38xuRwgHnY/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id=" + url.split("/")[-2]

# ===== Load teacher test CSV =====

test_df = pd.read_csv(path)

In [4]:
# ===== Use test data WITHOUT dropping Id =====
X_teacher_test = test_df.copy()

# ===== Predict =====
test_pred_log = best_model.predict(X_teacher_test)

# ===== Convert back =====
test_pred = np.expm1(test_pred_log)

# ===== Build submission =====
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_pred
})

# ===== Save =====
submission.to_csv("submission_SKB_SGD_R.csv", index=False)

submission.head()

,Id,SalePrice
0,1461,118352.679013
1,1462,148846.566640
2,1463,175538.239338
3,1464,198349.707818
4,1465,193439.997540


In [5]:
from google.colab import files
files.download("submission_SKB_SGD_R.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>